# Import libraries

In [9]:
%pip install gdown
import pandas as pd
import zipfile
import gdown

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC




Note: you may need to restart the kernel to use updated packages.


# Load Covid dataset

In [10]:
# Download dataset from Google Drive
file_id = "15VdFMp8JfCdjxOVA062qiykMekD__Adh"

gdown.download(
    id=file_id,
    output="Covid.zip",
    quiet=False
)

# Load datasets from ZIP file
with zipfile.ZipFile("Covid.zip", "r") as zf:
    df_train = pd.read_csv(
        zf.open("Corona_NLP_train.csv"),
        encoding="latin1"
    )

    df_test = pd.read_csv(
        zf.open("Corona_NLP_test.csv"),
        encoding="latin1"
    )

print("Corona Train Dataset Loaded:")
print(df_train.head())
print(df_train.shape)

print("Corona Test Dataset Loaded:")
print(df_test.head())
print(df_test.shape)



Downloading...
From: https://drive.google.com/uc?id=15VdFMp8JfCdjxOVA062qiykMekD__Adh
To: c:\Users\Probook G8\Desktop\Assignments\COMP723_Assignment\Covid.zip
100%|██████████| 4.60M/4.60M [00:00<00:00, 5.69MB/s]


Corona Train Dataset Loaded:
   UserName  ScreenName   Location     TweetAt  \
0      3799       48751     London  16-03-2020   
1      3800       48752         UK  16-03-2020   
2      3801       48753  Vagabonds  16-03-2020   
3      3802       48754        NaN  16-03-2020   
4      3803       48755        NaN  16-03-2020   

                                       OriginalTweet           Sentiment  
0  @MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...             Neutral  
1  advice Talk to your neighbours family to excha...            Positive  
2  Coronavirus Australia: Woolworths to give elde...            Positive  
3  My food stock is not the only one which is emp...            Positive  
4  Me, ready to go at supermarket during the #COV...  Extremely Negative  
(41157, 6)
Corona Test Dataset Loaded:
   UserName  ScreenName             Location     TweetAt  \
0         1       44953                  NYC  02-03-2020   
1         2       44954          Seattle, WA  02-03-2020   
2

In [11]:
# Check missing values
print("\nMissing values for training set:")
print(df_train[["OriginalTweet", "Sentiment"]].isnull().sum())
print("\nMissing values for testing set:")
print(df_test[["OriginalTweet", "Sentiment"]].isnull().sum())

# Check duplicate rows
print("\nNumber of duplicate rows in traning set:", df_train.duplicated().sum())
print("Number of duplicate rows in testing set:", df_test.duplicated().sum())

print("\ntraining dataset shape:", df_train.shape)
print("Testing dataset shape:", df_test.shape)

print("\nTraining sentiment counts:")
print(df_train["Sentiment"].value_counts())

print("\nTraining sentiment percentages:")
print(
    (df_train["Sentiment"].value_counts(normalize=True) * 100)
    .round(2)
)


Missing values for training set:
OriginalTweet    0
Sentiment        0
dtype: int64

Missing values for testing set:
OriginalTweet    0
Sentiment        0
dtype: int64

Number of duplicate rows in traning set: 0
Number of duplicate rows in testing set: 0

training dataset shape: (41157, 6)
Testing dataset shape: (3798, 6)

Training sentiment counts:
Sentiment
Positive              11422
Negative               9917
Neutral                7713
Extremely Positive     6624
Extremely Negative     5481
Name: count, dtype: int64

Training sentiment percentages:
Sentiment
Positive              27.75
Negative              24.10
Neutral               18.74
Extremely Positive    16.09
Extremely Negative    13.32
Name: proportion, dtype: float64


#Select the text and category columns

In [12]:
#Select feature column for X and target colum for Y
X_train = df_train["OriginalTweet"]
X_test = df_test["OriginalTweet"]

y_train = df_train["Sentiment"]
y_test = df_test["Sentiment"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (41157,)
X_test shape: (3798,)
y_train shape: (41157,)
y_test shape: (3798,)


# Convert text into numerial features

In [13]:
vectorizer = TfidfVectorizer(max_features=3000)

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

#Train the Naïve Bayes model, test the model and evaluate the results

In [14]:
# Train the Naïve Bayes model
model = MultinomialNB()
model.fit(X_train_vectorized, y_train)

# Test the model and evaluate the results
y_pred = model.predict(X_test_vectorized)
print(classification_report(y_test, y_pred))

                    precision    recall  f1-score   support

Extremely Negative       0.72      0.15      0.25       592
Extremely Positive       0.78      0.21      0.33       599
          Negative       0.43      0.53      0.47      1041
           Neutral       0.63      0.40      0.49       619
          Positive       0.37      0.72      0.49       947

          accuracy                           0.45      3798
         macro avg       0.59      0.40      0.41      3798
      weighted avg       0.55      0.45      0.42      3798



# Fine tuning the Naïve Bayes model

In [15]:
# Build a Naïve Bayes model pipeline
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=3000)),
    ("nb", MultinomialNB())
])

param_grid = {
    "nb__alpha": [0.01, 0.1, 0.5, 1.0, 2.0]
}

grid_search = GridSearchCV(
    nb_pipeline,
    param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV macro F1:", round(grid_search.best_score_, 4))

# Evaluate the selected model on the test set once
y_pred = grid_search.best_estimator_.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Best parameters: {'nb__alpha': 0.01}
Best CV macro F1: 0.427
                    precision    recall  f1-score   support

Extremely Negative       0.68      0.19      0.29       592
Extremely Positive       0.74      0.25      0.37       599
          Negative       0.43      0.54      0.47      1041
           Neutral       0.64      0.42      0.51       619
          Positive       0.38      0.68      0.49       947

          accuracy                           0.45      3798
         macro avg       0.57      0.41      0.43      3798
      weighted avg       0.54      0.45      0.44      3798



# Train the Neural Network model, test and evaluate the results

In [16]:
# Build a Neural Network pipeline
model_nn = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation="relu",
    solver="adam",
    max_iter=100,
    random_state=42
)

model_nn.fit(X_train_vectorized, y_train)

# Make predictions
y_pred_nn = model_nn.predict(X_test_vectorized)

# Evaluate the results
print("Neural Network Results")
print(
    classification_report(
        y_test,
        y_pred_nn,
        zero_division=0
    )
)

Neural Network Results
                    precision    recall  f1-score   support

Extremely Negative       0.59      0.59      0.59       592
Extremely Positive       0.63      0.60      0.61       599
          Negative       0.53      0.55      0.54      1041
           Neutral       0.66      0.61      0.64       619
          Positive       0.51      0.54      0.52       947

          accuracy                           0.57      3798
         macro avg       0.58      0.58      0.58      3798
      weighted avg       0.57      0.57      0.57      3798



c:\Users\Probook G8\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


# Fine tuning the Neural Network

In [17]:
# Build a Neural Network pipeline
nn_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=3000)),
    ("nn", MLPClassifier(
        activation="relu",
        solver="adam",
        max_iter=100,
        early_stopping=False,
        random_state=42
    ))
])

# Set up fine tuning parameters
param_grid_nn = {
    "nn__hidden_layer_sizes": [(50,), (100,),(50,100)],
    "nn__alpha": [0.0001]
}

grid_search_nn = GridSearchCV(
    nn_pipeline,
    param_grid_nn,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search_nn.fit(X_train, y_train)

print("Best parameters:", grid_search_nn.best_params_)
print("Best CV macro F1:", round(grid_search_nn.best_score_, 4))

y_pred_nn = grid_search_nn.best_estimator_.predict(X_test)

print("Neural Network — Fine-tuned Results")
print(classification_report(
    y_test, y_pred_nn, zero_division=0
))

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best parameters: {'nn__alpha': 0.0001, 'nn__hidden_layer_sizes': (50, 100)}
Best CV macro F1: 0.5796
Neural Network — Fine-tuned Results
                    precision    recall  f1-score   support

Extremely Negative       0.62      0.60      0.61       592
Extremely Positive       0.65      0.62      0.64       599
          Negative       0.54      0.53      0.53      1041
           Neutral       0.61      0.68      0.64       619
          Positive       0.54      0.54      0.54       947

          accuracy                           0.58      3798
         macro avg       0.59      0.59      0.59      3798
      weighted avg       0.58      0.58      0.58      3798



# Train the SVM (Support Vector Machine) model, test the model and evaluate the results

In [18]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# Create the baseline SVM model
model_svm = LinearSVC(
    random_state=42
)

# Train the model
model_svm.fit(X_train_vectorized, y_train)

# Make predictions
y_pred_svm = model_svm.predict(X_test_vectorized)

# Evaluate the model
print("Baseline SVM Results")
print(classification_report(
    y_test,
    y_pred_svm,
    zero_division=0
))

Baseline SVM Results
                    precision    recall  f1-score   support

Extremely Negative       0.59      0.57      0.58       592
Extremely Positive       0.64      0.65      0.65       599
          Negative       0.52      0.45      0.48      1041
           Neutral       0.61      0.72      0.66       619
          Positive       0.50      0.51      0.51       947

          accuracy                           0.56      3798
         macro avg       0.57      0.58      0.58      3798
      weighted avg       0.56      0.56      0.56      3798



# Fine tuning SVM model

In [19]:
# Create the SVM pipeline
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=3000)),
    ("svm", LinearSVC(random_state=42, max_iter=3000))
])

# Parameters to test
param_grid_svm = {
    "svm__C": [0.1, 1.0, 10.0]
}

# Tune using cross-validation
grid_search_svm = GridSearchCV(
    svm_pipeline,
    param_grid_svm,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search_svm.fit(X_train, y_train)

print("Best parameters:", grid_search_svm.best_params_)
print(
    "Best CV macro F1:",
    round(grid_search_svm.best_score_, 4)
)

# Final test-set evaluation
y_pred_svm = grid_search_svm.best_estimator_.predict(X_test)

print("Fine-tuned SVM Results")
print(classification_report(
    y_test,
    y_pred_svm,
    zero_division=0
))

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best parameters: {'svm__C': 1.0}
Best CV macro F1: 0.5545
Fine-tuned SVM Results
                    precision    recall  f1-score   support

Extremely Negative       0.59      0.57      0.58       592
Extremely Positive       0.64      0.65      0.65       599
          Negative       0.52      0.45      0.48      1041
           Neutral       0.61      0.72      0.66       619
          Positive       0.50      0.51      0.51       947

          accuracy                           0.56      3798
         macro avg       0.57      0.58      0.58      3798
      weighted avg       0.56      0.56      0.56      3798

